In [32]:
# Imports

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

In [33]:
# Configuration (change behavour without touching core logic)

TRAIN_PATH = "data_train.csv"
TEST_PATH = "smiles_test.csv"
SAMPLE_SUB_PATH = "sample_submission.csv"

OUTPUT_SUB_PATH = "submission_2nd_script_randomforest2.csv"
RANDOM_STATE = 42
N_SPLITS = 5

# Feature config
USE_MORGAN = True
MORGAN_RADIUS = 2
MORGAN_BITS = 2048

USE_RDKit_DESCRIPTORS = True

# Model selection
MODEL_NAME = "rf" # options: "logreg", "extratrees", "rf"

In [34]:
# Feature engineering

DESCRIPTOR_FUNCS = {
    "MolWt": Descriptors.MolWt, # molecular weight
    "MolLogP": Descriptors.MolLogP, # logP / lipophilicity (hwo well it binds to fat)
    "TPSA": rdMolDescriptors.CalcTPSA, # topological polar surface area
    "NumHDonors": rdMolDescriptors.CalcNumHBD, # number of H-bond donors
    "NumHAcceptors": rdMolDescriptors.CalcNumHBA, # number of H-bond acceptors
    "NumRotatableBonds": rdMolDescriptors.CalcNumRotatableBonds, # conformational flexibility
    "RingCount": rdMolDescriptors.CalcNumRings, # number of rings
    "HeavyAtomCount": rdMolDescriptors.CalcNumHeavyAtoms, # number of non-hydrogen atoms
    "FractionCSP3": rdMolDescriptors.CalcFractionCSP3, # rough measure of 3D character / saturation
}

def mol_from_smiles(smiles):
    """parse SMILES - return mol or none if invalid"""
    if pd.isna(smiles):
        return None
    try:
        mol = Chem.MolFromSmiles(smiles)
        return mol
    except Exception:
        return None

def morgan_fp_from_mol(mol, radius=MORGAN_RADIUS, n_bits=MORGAN_BITS):
    """return Morgan fingerprint as numpy array"""
    arr = np.zeros((n_bits,), dtype=np.float32)
    if mol is None:
        return arr
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def descriptors_from_mol(mol):
    """return selected RDKit descriptors as numpy array"""
    vals = []
    for name, func in DESCRIPTOR_FUNCS.items():
        try:
            if mol is None:
                vals.append(0.0)
            else:
                vals.append(float(func(mol)))
        except Exception:
            vals.append(0.0)
    return np.array(vals, dtype=np.float32)

def build_feature_matrix(smiles_series, use_morgan=USE_MORGAN, use_desc=True,
                         radius=MORGAN_RADIUS, n_bits=MORGAN_BITS):
    """convert a column of SMILES strings into a ML feature matrix"""

    # parse all SMILES strings into molecule objects
    mols = [mol_from_smiles(s) for s in smiles_series]

    features = []

    # generate Morgan fingerprints if requested
    if use_morgan:
        fps = np.vstack([morgan_fp_from_mol(m, radius=radius, n_bits=n_bits) for m in mols])
        features.append(fps)

    # generate descriptor matrix if requested
    if use_desc:
        desc = np.vstack([descriptors_from_mol(m) for m in mols])
        features.append(desc)

    # return either a single block or concat blocks
    if len(features) == 1:
        return features[0]
    return np.hstack(features)

In [35]:
# Model selection and factory

def get_model(model_name):
    """return a model based on the selection (RF worked best)"""
    if model_name == "logreg":
        return LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            solver="liblinear",
            random_state=RANDOM_STATE,
        )
    elif model_name == "extratrees":
        return ExtraTreesClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    elif model_name == "rf":
        return RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    else:
        raise ValueError(f"Unknown MODEL_NAME: {model_name}")

In [36]:
# Training & Evaluation

def train_multitask_models(X_train, X_test, train_df, task_cols, n_splits=5):
    """train one separate binary model per task using cross-validation"""
    oof_preds = pd.DataFrame(index=train_df.index, columns=task_cols, dtype=float)
    test_preds = pd.DataFrame(index=np.arange(X_test.shape[0]), columns=task_cols, dtype=float)

    for task in task_cols:
        print(f"\nTraining {task} ...")

        y_raw = train_df[task].values

        # keep only samples with known labels for this task
        known_mask = y_raw != 0

        # subset features and labels for this task
        X_task = X_train[known_mask]
        y_task = y_raw[known_mask]

        # convert labels: +1 becomes 1, -1 becomes 0
        y_task = (y_task == 1).astype(int)

        # store the OG training indices so OOF preds can be written back
        known_indices = train_df.index[known_mask]

        # fallback if too few positives/negatives
        n_pos = int((y_task == 1).sum())
        n_neg = int((y_task == 0).sum())
        print(f"  known={len(y_task)}, pos={n_pos}, neg={n_neg}")

        # reduce N of folds if the task has too less pos. or neg. samples
        if n_pos < n_splits or n_neg < n_splits:
            print("  Warning: too few samples for StratifiedKFold, using fewer folds.")
            effective_splits = max(2, min(n_splits, n_pos, n_neg))
        else:
            effective_splits = n_splits

        # SKF preserves class ratios approximately in each fold
        skf = StratifiedKFold(
            n_splits=effective_splits,
            shuffle=True,
            random_state=RANDOM_STATE
        )

        fold_test_preds = []
        
        # Cross-Val loop for current task
        for fold, (tr_idx, va_idx) in enumerate(skf.split(X_task, y_task), start=1):
            # split current task-specific data into train/validation fold
            X_tr, X_va = X_task[tr_idx], X_task[va_idx]
            y_tr, y_va = y_task[tr_idx], y_task[va_idx]

            # create and fit model on train fold
            model = get_model(MODEL_NAME)
            model.fit(X_tr, y_tr)

            # predictions for val fold and test set
            va_pred = model.predict_proba(X_va)[:, 1]
            te_pred = model.predict_proba(X_test)[:, 1]

            # compute AUC for monitoring
            fold_auc = roc_auc_score(y_va, va_pred)
            print(f"  fold {fold}: AUC={fold_auc:.5f}")

            # write val predictions back into the correct rows and store test predictions for later
            oof_preds.loc[known_indices[va_idx], task] = va_pred
            fold_test_preds.append(te_pred)

        # final test prediction = mean across folds
        test_preds[task] = np.mean(fold_test_preds, axis=0)

    return oof_preds, test_preds

def compute_mean_auc(train_df, oof_df, task_cols):
    """compute the mean AUC across all tasks - like in the provided script"""
    aucs = []
    print("\nPer-task OOF AUC:")
    for task in task_cols:
        y = train_df[task].values

        # only use samples with known labels for this task
        mask = y != 0  

        # convert OG labels to binary labels: +1 -> 1 (active), -1 -> 0 (inactive)
        y_true = (y[mask] == 1).astype(int)
        # get the model scores for the same subset
        y_score = oof_df.loc[train_df.index[mask], task].values.astype(float)

        # calculate AUC
        auc = roc_auc_score(y_true, y_score)
        aucs.append(auc)
        print(f"{task:>6}: {auc:.5f}")

    # calculate mean AUC from all tasks
    mean_auc = float(np.mean(aucs))
    print(f"\nMean OOF AUC: {mean_auc:.5f}")
    return mean_auc, aucs

In [37]:
# Main

def main():
    print("Loading data...")

    # load CSVs
    train_df = pd.read_csv(TRAIN_PATH, index_col=0)
    test_df = pd.read_csv(TEST_PATH, index_col=0)
    sample_sub = pd.read_csv(SAMPLE_SUB_PATH, index_col=0)

    # sanity check
    assert "smiles" in train_df.columns, "Train file must contain 'smiles'"
    assert "smiles" in test_df.columns, "Test file must contain 'smiles'"
    
    # get task cols
    task_cols = [c for c in train_df.columns if c.startswith("task")]
    print(f"Tasks: {task_cols}")

    print("\nBuilding features...")
    X_train = build_feature_matrix(
        train_df["smiles"],
        use_morgan=USE_MORGAN,
        use_desc=USE_RDKit_DESCRIPTORS,
        radius=MORGAN_RADIUS,
        n_bits=MORGAN_BITS,
    )
    X_test = build_feature_matrix(
        test_df["smiles"],
        use_morgan=USE_MORGAN,
        use_desc=USE_RDKit_DESCRIPTORS,
        radius=MORGAN_RADIUS,
        n_bits=MORGAN_BITS,
    )

    print(f"X_train shape: {X_train.shape}")
    print(f"X_test  shape: {X_test.shape}")

    print("\nTraining models with CV...")
    oof_preds, test_preds = train_multitask_models(
        X_train=X_train,
        X_test=X_test,
        train_df=train_df,
        task_cols=task_cols,
        n_splits=N_SPLITS
    )

    print("\nEvaluating local CV...")
    compute_mean_auc(train_df, oof_preds, task_cols)

    print("\nBuilding submission...")
    submission = sample_sub.copy()

    # fill each task column with predicted probabilities
    for task in task_cols:
        submission[task] = test_preds[task].values

    submission.to_csv(OUTPUT_SUB_PATH)
    print(f"Saved submission to: {OUTPUT_SUB_PATH}")

if __name__ == "__main__":
    main()

Loading data...
Tasks: ['task1', 'task2', 'task3', 'task4', 'task5', 'task6', 'task7', 'task8', 'task9', 'task10', 'task11']

Building features...


[16:37:38] WARNING: not removing hydrogen atom without neighbors
[16:37:38] WARNING: not removing hydrogen atom without neighbors
[16:37:39] WARNING: not removing hydrogen atom without neighbors
[16:37:39] WARNING: not removing hydrogen atom without neighbors
[16:37:52] WARNING: not removing hydrogen atom without neighbors
[16:37:52] WARNING: not removing hydrogen atom without neighbors
[16:37:53] WARNING: not removing hydrogen atom without neighbors
[16:37:53] WARNING: not removing hydrogen atom without neighbors


X_train shape: (12000, 2057)
X_test  shape: (5896, 2057)

Training models with CV...

Training task1 ...
  known=3626, pos=624, neg=3002
  fold 1: AUC=0.87673
  fold 2: AUC=0.90319
  fold 3: AUC=0.89549
  fold 4: AUC=0.90821
  fold 5: AUC=0.92115

Training task2 ...
  known=953, pos=73, neg=880
  fold 1: AUC=0.77216
  fold 2: AUC=0.84981
  fold 3: AUC=0.80833
  fold 4: AUC=0.71611
  fold 5: AUC=0.75244

Training task3 ...
  known=962, pos=637, neg=325
  fold 1: AUC=0.59681
  fold 2: AUC=0.61989
  fold 3: AUC=0.61333
  fold 4: AUC=0.63634
  fold 5: AUC=0.65058

Training task4 ...
  known=1010, pos=60, neg=950
  fold 1: AUC=0.51031
  fold 2: AUC=0.62961
  fold 3: AUC=0.55154
  fold 4: AUC=0.59956
  fold 5: AUC=0.64912

Training task5 ...
  known=610, pos=46, neg=564
  fold 1: AUC=0.87365
  fold 2: AUC=0.77532
  fold 3: AUC=0.84710
  fold 4: AUC=0.81023
  fold 5: AUC=0.67187

Training task6 ...
  known=1023, pos=462, neg=561
  fold 1: AUC=0.91006
  fold 2: AUC=0.88484
  fold 3: AUC=0.9127